In [1]:
import ee
import geemap
import matplotlib.pyplot as plt

#Initialize earth Engine 
ee.Initialize(project="drishtisar")
print("Google Earth Engine initialized successfully!")

c:\Users\LENOVO\anaconda3\envs\drishtisar\lib\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Google Earth Engine initialized successfully!


In [2]:
# Cell 2: Define Area of Interest (AOI) - Flexible version

# Option 1: Pre-defined areas (easy to switch)
areas = {
    # [min_lon, min_lat, max_lon, max_lat] #We can get the coorrdinates by using bbox finder 
    "Assam_Flood": [92.8, 26.0, 93.2, 26.4],
    "North_Bihar": [85.5, 25.8, 87.0, 26.5],
    "Kerala": [76.0, 9.5, 77.0, 10.5],
    "Uttarakhand": [78.5, 29.8, 79.5, 30.5],
    "Delhi_NCR": [76.8, 28.4, 77.5, 28.9]
}

selected_area = "Uttarakhand"   # Change this string to test other areas

aoi_coords = areas[selected_area]
aoi = ee.Geometry.Rectangle(aoi_coords)

# Center point for map (average of coordinates)
center_lon = (aoi_coords[0] + aoi_coords[2]) / 2
center_lat = (aoi_coords[1] + aoi_coords[3]) / 2
center = [center_lat, center_lon]

print(f"Selected AOI: {selected_area}")
print(f"Coordinates: {aoi_coords}")

Selected AOI: Uttarakhand
Coordinates: [78.5, 29.8, 79.5, 30.5]


In [ ]:
# Cell 3: Load Sentinel-2 (Optical - Colour) - Recent cloudy possible image
s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(aoi)
      .filterDate('2025-07-01', '2025-08-31')   # Monsoon period - likely cloudy
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80))  # Less than 80% cloud
      .sort('CLOUDY_PIXEL_PERCENTAGE') # Sorts the image so the least cloudy ones comes first
      .first())  # Take the least cloudy image

# True colour visualization (RGB)
vis_s2 = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}

print("Sentinel-2 image loaded")

Sentinel-2 image loaded


In [ ]:
# Cell 4: Load Sentinel-1 (SAR - All weather)
s1 = (ee.ImageCollection('COPERNICUS/S1_GRD') #Ground range detected image collection
      .filterBounds(aoi)
      .filterDate('2025-07-01', '2025-08-31')
      .filter(ee.Filter.eq('instrumentMode', 'IW')) #Keeps only images collected in Interferometric Wide swath mode(Primary mode for s1)
      .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING'))  # or DESCENDING #in s1 it tells which way the satellite was moving when it collected that scene (ASC- South to north), DESC - N to S
      .select(['VV', 'VH']) #VV is vertical transmit, vertical receive #VH is vertical transmit, horizontal receive 
      .first())

#Polarization is VH and VV which is how radar signals are transmitted and received 

# Visualization for SAR (VV polarization - good for water/flood)
vis_s1 = {'bands': 'VV', 'min': -25, 'max': 0}

print("Sentinel-1 SAR image loaded")

Sentinel-1 SAR image loaded


In [10]:
# Cell 5: Side-by-side comparison view

Map = geemap.Map(center=center, zoom=11)

# Convert the Earth Engine images into tile layers for the split map
left_layer = geemap.ee_tile_layer(s2, vis_s2, 'Sentinel-2 (Optical)', shown=True, opacity=1.0)
right_layer = geemap.ee_tile_layer(s1, vis_s1, 'Sentinel-1 (SAR)', shown=True, opacity=0.85)

# Show Sentinel-2 on the left and Sentinel-1 on the right
Map.split_map(
    left_layer=left_layer,
    right_layer=right_layer,
    left_label='Sentinel-2 (Optical)',
    right_label='Sentinel-1 (SAR)',
    layer_control=False
)

Map

Map(center=[30.15, 79.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…